# 第 12 课｜PyTorch 神经网络模块与多层感知机（MLP）

**本课主线：从手动管理参数，走向可复用的神经网络模块。**

上一课你已经实现了 `loss.backward()` 与参数更新；这节课要让 PyTorch 自动管理许多参数，同时弄清楚**为什么叠加线性层仍然不够、为什么需要非线性激活函数**。

学习目标：

1. 解释 `nn.Module`、`nn.Linear`、`nn.Parameter` 和 `forward()` 的作用。
2. 说出 `nn.Linear(in_features, out_features)` 的权重形状和输出形状。
3. 使用 `ReLU` 搭建两层 MLP，训练解决 XOR 二分类问题。
4. 区分 **logits / 概率、train / eval、梯度计算 / 参数更新**。
5. 保存 `state_dict()` 并重载模型；把 MLP 与 Transformer 的 FFN 联系起来。

**建议用时：2–3 小时。** 全程仅需 PyTorch，默认 CPU，不用下载预训练模型或使用 RTX 3060。练习与参考答案位于后半部分；请先自行完成。

## 0. 环境检查

若缺少 PyTorch，可在终端执行：

```bash
python -m pip install torch jupyterlab ipykernel
```

使用 3060 运行 CUDA 时需要安装合适的 GPU 版 PyTorch；可用 [官方安装选择器](https://pytorch.org/get-started/locally/) 操作。今天只用 CPU，可以暂时不处理显卡环境。

In [2]:
import json
from pathlib import Path

import torch
from torch import nn

# 小矩阵运算中限制线程数量，减少部分 CPU 环境里的调度开销。
if torch.get_num_threads() > 4:
    torch.set_num_threads(4)

torch.manual_seed(2026)
DEVICE = torch.device("cpu")
print("PyTorch:", torch.__version__)
print("本课设备:", DEVICE)
print("CUDA 可用:", torch.cuda.is_available())

PyTorch: 2.9.1+cu126
本课设备: cpu
CUDA 可用: True


## 1. `nn.Linear`：神经网络里最常见的可学习变换

一个线性层（严格说是仿射变换）执行：

$
Y = XW^\top + b
$

PyTorch 定义 `nn.Linear(in_features=2, out_features=3)` 时：

- 输入 `X` 的最后一维是 **2**；
- 权重 `W` 形状是 **`(3, 2)`**；
- 偏置 `b` 形状是 **`(3,)`**；
- 输出最后一维变成 **3**。

注意：`nn.Linear` 的 `weight` 是 `[输出维, 输入维]`，不是 `[输入维, 输出维]`。

In [3]:
x = torch.tensor([[1., 2.], [3., 4.]])  # [2, 2]
layer = nn.Linear(2, 3)

# 将权重设置为已知数值，方便手动核对。
with torch.no_grad():
    layer.weight.copy_(torch.tensor([[1., 0.], [0., 1.], [1., -1.]]))
    layer.bias.copy_(torch.tensor([0.5, -0.5, 1.0]))

output = layer(x)
by_formula = x @ layer.weight.T + layer.bias

print("X:", tuple(x.shape))
print("W:", tuple(layer.weight.shape))
print("b:", tuple(layer.bias.shape))
print("Y:", tuple(output.shape))
print("输出：", output)
print("与手算公式一致：", torch.allclose(output, by_formula))
torch.testing.assert_close(output, by_formula)

X: (2, 2)
W: (3, 2)
b: (3,)
Y: (2, 3)
输出： tensor([[1.5000, 1.5000, 0.0000],
        [3.5000, 3.5000, 0.0000]], grad_fn=<AddmmBackward0>)
与手算公式一致： True


### `nn.Linear` 对张量的最后一维工作

输入不必只有二维。以后 Transformer 常见的输入是 `[B, T, D]`：批量大小、序列长度、隐藏维度。

只要最后一维是 `in_features`，`nn.Linear` 就会对各个位置应用**同一组权重**。

In [4]:
token_features = torch.zeros(2, 3, 2)   # [B=2, T=3, D=2]
token_outputs = layer(token_features)      # [2, 3, 3]
print("输入形状:", tuple(token_features.shape))
print("输出形状:", tuple(token_outputs.shape))
assert token_outputs.shape == (2, 3, 3)

输入形状: (2, 3, 2)
输出形状: (2, 3, 3)


## 2. 激活函数：为什么两层 `Linear` 还不够？

如果中间没有非线性激活：

$
(W_2(W_1x+b_1)+b_2) = (W_2W_1)x + (W_2b_1+b_2)
$

右侧仍是一层仿射变换，表达能力没有本质提升。`ReLU(z)=max(0,z)` 给模型引入非线性。之后的 Transformer FFN 也有类似的非线性步骤（现代模型还可能使用 GELU、SwiGLU 等）。

In [5]:
values = torch.tensor([-2., -0.5, 0., 1., 3.])
print("原始值:", values.tolist())
print("ReLU 后:", nn.ReLU()(values).tolist())

first = nn.Linear(2, 5)
second = nn.Linear(5, 3)
sample = torch.tensor([[1., 2.], [-2., 0.5]])

# 没有激活函数：两层可以合成一个线性层。
two_layers = second(first(sample))
combined_W = second.weight @ first.weight
combined_b = second.weight @ first.bias + second.bias
one_layer_equivalent = sample @ combined_W.T + combined_b
print("两层无激活可以合并：", torch.allclose(two_layers, one_layer_equivalent))
assert torch.allclose(two_layers, one_layer_equivalent)

原始值: [-2.0, -0.5, 0.0, 1.0, 3.0]
ReLU 后: [0.0, 0.0, 0.0, 1.0, 3.0]
两层无激活可以合并： True


## 3. `nn.Module` / `nn.Parameter` / buffer 各负责什么？

- `nn.Module` 是模型或层的容器：注册子模块、参数，提供 `parameters()`、`state_dict()`、`train()`、`eval()` 等方法。
- 将 `nn.Parameter` 赋给模块属性，会把它注册为**可学习参数**。
- `register_buffer` 注册**模型状态但不是可训练参数**，例如某些归一化层的统计量。

下面的 `scale` 会接收梯度，`offset` 不会。

In [6]:
class ScaleAndOffset(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(2.0))
        self.register_buffer("offset", torch.tensor(1.0))

    def forward(self, x):
        return x * self.scale + self.offset

small_module = ScaleAndOffset()
print("可学习参数:", list(dict(small_module.named_parameters())))
print("buffers:", list(dict(small_module.named_buffers())))
print("state_dict 的键:", list(small_module.state_dict()))

result = small_module(torch.tensor([1., 2.]))
(result**2).sum().backward()
print("scale.grad:", small_module.scale.grad.item())
print("offset.grad:", small_module.offset.grad)
assert small_module.scale.grad is not None
assert small_module.offset.grad is None

可学习参数: ['scale']
buffers: ['offset']
state_dict 的键: ['scale', 'offset']
scale.grad: 26.0
offset.grad: None


## 4. 写出第一个真正的 MLP

我们设计：

```text
输入 [N, 2]
    ↓ nn.Linear(2, 16)
隐藏层 [N, 16]
    ↓ nn.ReLU()
隐藏层 [N, 16]
    ↓ nn.Linear(16, 1)
输出 logits [N, 1]
```

`forward()` 定义一次前向计算；`model(x)` 会调用它。输出是 **logits 分数**，不是 0–1 概率。

In [7]:
class XORMLP(nn.Module):
    def __init__(self, hidden_features=16):
        super().__init__()
        self.linear1 = nn.Linear(2, hidden_features)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(hidden_features, 1)

    def forward(self, x):
        hidden = self.activation(self.linear1(x))
        return self.linear2(hidden)

model = XORMLP(16)
example = torch.tensor([[0., 0.], [1., 0.], [0., 1.]])
logits = model(example)
probabilities = torch.sigmoid(logits)
print(model)
print("logits.shape:", tuple(logits.shape))
print("logits:", logits.detach())
print("概率:", probabilities.detach())
print("参数总数:", sum(p.numel() for p in model.parameters()))
assert logits.shape == (3, 1)
assert sum(p.numel() for p in model.parameters()) == 65

XORMLP(
  (linear1): Linear(in_features=2, out_features=16, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=16, out_features=1, bias=True)
)
logits.shape: (3, 1)
logits: tensor([[-0.1142],
        [-0.0135],
        [-0.2658]])
概率: tensor([[0.4715],
        [0.4966],
        [0.4339]])
参数总数: 65


### 两种搭建网络的方法

对于简单的顺序结构，`nn.Sequential` 也可以表达同一层次结构：

```python
nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,1))
```

`nn.Module` 自定义类更适合需要复杂 `forward()`、分支和额外状态的模型。

In [8]:
sequential_mlp = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
print("Sequential 输出形状:", tuple(sequential_mlp(example).shape))

Sequential 输出形状: (3, 1)


## 5. XOR：为什么需要非线性？

我们用两个数作为特征，标签按下面规则确定：

| 输入 | 标签 |
|---|---|
| `(0, 0)` | 0 |
| `(0, 1)` | 1 |
| `(1, 0)` | 1 |
| `(1, 1)` | 0 |

这个任务叫 XOR（异或）。用**一条直线作为决策边界的线性分类器**无法同时把四个角点全部正确分开；具有非线性的 MLP 则可以。我们在四个角附近加少量随机噪声，分别用不同随机种子生成训练集和测试集。

In [9]:
def make_xor_data(samples_per_corner=64, noise=0.12, seed=7):
    corners = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
    labels = torch.tensor([[0.], [1.], [1.], [0.]])
    count = 4 * samples_per_corner
    generator = torch.Generator().manual_seed(seed)
    x = corners.repeat_interleave(samples_per_corner, dim=0)
    x = x + noise * torch.randn(count, 2, generator=generator)
    y = labels.repeat_interleave(samples_per_corner, dim=0)
    permutation = torch.randperm(count, generator=generator)
    return x[permutation], y[permutation]

train_x, train_y = make_xor_data(seed=7)
test_x, test_y = make_xor_data(seed=8)
print("训练输入/标签:", tuple(train_x.shape), tuple(train_y.shape))
print("测试输入/标签:", tuple(test_x.shape), tuple(test_y.shape))
print("前四条数据示例:", train_x[:4].tolist())
print("对应标签:", train_y[:4].squeeze(-1).tolist())
assert train_x.shape == (256, 2) and train_y.shape == (256, 1)

训练输入/标签: (256, 2) (256, 1)
测试输入/标签: (256, 2) (256, 1)
前四条数据示例: [[-0.08081483095884323, 0.9145042896270752], [1.015587329864502, 1.03000009059906], [0.8144152760505676, 0.0012645815731957555], [0.11485225707292557, -0.07942410558462143]]
对应标签: [1.0, 0.0, 1.0, 0.0]


## 6. 损失函数：为什么用 `BCEWithLogitsLoss`？

本题是**二分类**，标签只有 0/1。使用：

```python
loss_fn = nn.BCEWithLogitsLoss()
loss = loss_fn(logits, labels)
```

它内部已经包含 sigmoid 相关计算，比手动 `sigmoid -> BCELoss` 更稳定。

**重要：损失函数输入的是模型直接输出的 logits，不能先对 logits 做 sigmoid 再传给 `BCEWithLogitsLoss`。**

`torch.sigmoid(logits)` 主要用于观察概率或推理阶段把分数变成 0–1 范围。

In [10]:
loss_fn = nn.BCEWithLogitsLoss()
zero_logits = torch.tensor([[0.], [0.], [0.], [0.]])
four_labels = torch.tensor([[0.], [1.], [1.], [0.]])
print("全部 logits=0 时，概率:", torch.sigmoid(zero_logits).squeeze(-1).tolist())
print("此时的损失:", round(loss_fn(zero_logits, four_labels).item(), 6))
assert abs(loss_fn(zero_logits, four_labels).item() - 0.693147) < 0.00001

全部 logits=0 时，概率: [0.5, 0.5, 0.5, 0.5]
此时的损失: 0.693147


## 7. 先观察一次真实的参数更新

训练流程与第 11 课一致，但这次用 `model.parameters()` 自动收集所有层的参数：

1. `model.train()`：切换到训练模式；
2. `optimizer.zero_grad()`：清除上次梯度；
3. `model(train_x)`、损失函数：前向计算；
4. `loss.backward()`：计算梯度；
5. `optimizer.step()`：更新参数。

`train()/eval()` 与 `no_grad()` **不是同一回事**。前者调整某些层的行为（如 Dropout），后者控制是否构建求导计算图。

In [11]:
torch.manual_seed(2026)
one_step_model = XORMLP(16)
optimizer = torch.optim.Adam(one_step_model.parameters(), lr=0.05)

before = one_step_model.linear1.weight.detach().clone()
one_step_model.train()
optimizer.zero_grad()
one_step_logits = one_step_model(train_x)
one_step_loss = loss_fn(one_step_logits, train_y)
one_step_loss.backward()
print("本次损失:", round(one_step_loss.item(), 6))
print("第一层权重的梯度范数:", round(one_step_model.linear1.weight.grad.norm().item(), 6))
optimizer.step()
after = one_step_model.linear1.weight.detach().clone()
print("更新后权重变化：", not torch.equal(before, after))
assert not torch.equal(before, after)

本次损失: 0.693751
第一层权重的梯度范数: 0.054106
更新后权重变化： True


## 8. 定义完整的训练与评估函数

为了避免把**训练集表现**误当作**泛化表现**，分别使用训练集和独立测试集：

- `train_classifier` 只用训练集更新参数。
- `evaluate_classifier` 在测试集上评估，使用 `eval()` 与 `no_grad()`；它不会更新参数。

这里的合成数据很简单，分数只用于说明非线性的作用，不能代表实际大模型的能力。

In [12]:
def evaluate_classifier(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = nn.BCEWithLogitsLoss()(logits, y).item()
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        accuracy = (predictions == y).float().mean().item()
    return loss, accuracy


def train_classifier(model, x, y, epochs=220, lr=0.05):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    initial_loss, _ = evaluate_classifier(model, x, y)
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    final_loss, _ = evaluate_classifier(model, x, y)
    return initial_loss, final_loss

### 实验：线性基线与两层 MLP 正面对照

两种模型使用同一份训练集和测试集，优化方法都用 Adam，但模型结构不同：

- `nn.Linear(2,1)`：只有一个线性决策边界；
- `XORMLP(16)`：在两层线性变换中插入 ReLU。

预测一下：谁的测试准确率可能更高？然后运行。

In [13]:
torch.manual_seed(2026)
linear_baseline = nn.Linear(2, 1)
linear_initial, linear_final = train_classifier(linear_baseline, train_x, train_y, epochs=140)
linear_test_loss, linear_test_acc = evaluate_classifier(linear_baseline, test_x, test_y)

torch.manual_seed(2026)
trained_mlp = XORMLP(16)
mlp_initial, mlp_final = train_classifier(trained_mlp, train_x, train_y, epochs=220)
mlp_test_loss, mlp_test_acc = evaluate_classifier(trained_mlp, test_x, test_y)

print(f"线性模型: 训练 loss {linear_initial:.4f} -> {linear_final:.4f}; 测试准确率 {linear_test_acc:.2%}")
print(f"两层 MLP: 训练 loss {mlp_initial:.4f} -> {mlp_final:.4f}; 测试准确率 {mlp_test_acc:.2%}")
assert mlp_test_acc >= 0.94
assert mlp_test_acc > linear_test_acc + 0.1

线性模型: 训练 loss 0.7059 -> 0.6926; 测试准确率 51.17%
两层 MLP: 训练 loss 0.6938 -> 0.0009; 测试准确率 99.61%


### 回到 XOR 的四个角点

对四个标准角点观察预测概率，应该大致接近 `[0, 1, 1, 0]`。注意：概率不是标签，概率需要经过阈值（本例是 0.5）才能得到分类结果。

In [14]:
corners = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
trained_mlp.eval()
with torch.no_grad():
    corner_probs = torch.sigmoid(trained_mlp(corners)).squeeze(-1)
print("四个角点的预测概率：", [round(float(p), 4) for p in corner_probs])
print("对应预测类别：", (corner_probs >= 0.5).int().tolist())
assert (corner_probs >= 0.5).int().tolist() == [0, 1, 1, 0]

四个角点的预测概率： [0.0001, 0.9998, 1.0, 0.0001]
对应预测类别： [0, 1, 1, 0]


## 9. `model.eval()` 和 `torch.no_grad()` 不是一回事

我们用 `Dropout` 观察模式差异。训练模式下，Dropout 随机屏蔽部分值；评估模式下，它通常不屏蔽。`torch.no_grad()` 则是控制是否记录梯度，不会自动把 Dropout 切到评估模式。

In [15]:
torch.manual_seed(2026)
dropout = nn.Dropout(p=0.5)
ones = torch.ones(12)
dropout.train()
train_output = dropout(ones)
dropout.eval()
with torch.no_grad():
    eval_output = dropout(ones)
print("Dropout 训练模式：", train_output.tolist())
print("Dropout 评估模式：", eval_output.tolist())
print("评估模式是否全为 1：", torch.equal(eval_output, ones))
assert torch.equal(eval_output, ones)

Dropout 训练模式： [2.0, 0.0, 2.0, 2.0, 0.0, 0.0, 0.0, 2.0, 2.0, 2.0, 2.0, 0.0]
Dropout 评估模式： [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
评估模式是否全为 1： True


## 10. 保存和重新加载训练好的参数

推荐保存 `model.state_dict()`，不是把整个 Python 模型对象直接序列化。重载时先建**同样结构**的模型，再加载参数。

此单元会在 Notebook 的当前工作目录创建 `lesson_12_outputs/xor_mlp_state_dict.pt`。可查看 `Path.cwd()` 确认路径。

从不可信来源加载权重仍需谨慎；这里仅加载自己刚保存的文件，并显式使用 `weights_only=True`。

In [16]:
output_dir = Path("lesson_12_outputs")
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "xor_mlp_state_dict.pt"
torch.save(trained_mlp.state_dict(), model_path)

reloaded_mlp = XORMLP(16)
state = torch.load(model_path, map_location="cpu", weights_only=True)
reloaded_mlp.load_state_dict(state)
reloaded_mlp.eval()
trained_mlp.eval()
with torch.no_grad():
    original_output = trained_mlp(test_x)
    reloaded_output = reloaded_mlp(test_x)
print("文件：", model_path.resolve())
print("重载预测是否一致：", torch.allclose(original_output, reloaded_output))
assert torch.allclose(original_output, reloaded_output)

文件： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_12_outputs\xor_mlp_state_dict.pt
重载预测是否一致： True


## 11. 与 Transformer 的前馈网络（FFN）连接

Transformer 除了注意力，还包含前馈网络。一个常见的简单形式是：

$
\operatorname{FFN}(x)=W_2\,\mathrm{GELU}(W_1x+b_1)+b_2
$

这里省略了残差、归一化等模块，仅演示**对最后一维的变换**。

如果输入是 `[B,T,D]`，一个 `D → D_ff → D` 的 FFN 的输出仍是 `[B,T,D]`。它会逐位置处理 token；**只有 FFN 本身不负责让不同位置互相通信**，跨位置通信通常依靠注意力等机制。

In [17]:
class TokenFFN(nn.Module):
    def __init__(self, d_model=8, d_ff=24):
        super().__init__()
        self.up = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.down = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.down(self.activation(self.up(x)))

torch.manual_seed(123)
ffn = TokenFFN(d_model=8, d_ff=24)
features = torch.randn(2, 3, 8)
up_features = ffn.up(features)
ffn_output = ffn(features)
residual_output = features + ffn_output
print("输入：", tuple(features.shape))
print("中间升维：", tuple(up_features.shape))
print("FFN 输出：", tuple(ffn_output.shape))
print("加入残差之后：", tuple(residual_output.shape))
assert ffn_output.shape == features.shape

输入： (2, 3, 8)
中间升维： (2, 3, 24)
FFN 输出： (2, 3, 8)
加入残差之后： (2, 3, 8)


### 验证：FFN 是否会混合不同 token 的信息？

只把序列的最后一个 token 改成另一个值，再比较前两个 token 的输出。如果只有逐位置 FFN，前两个 token 的输出应当保持不变。随后计算一个简单损失，确认两个线性层的参数都接收梯度。

In [18]:
changed = features.clone()
changed[0, 2] += 100.0
changed_output = ffn(changed)
print("修改最后一个 token 后，前两个 token 的输出是否一致：",
      torch.allclose(ffn_output[0, :2], changed_output[0, :2]))

ffn.zero_grad()
(ffn_output**2).mean().backward()
for name, param in ffn.named_parameters():
    print(name, "shape=", tuple(param.shape), "梯度范数=", round(param.grad.norm().item(), 6))
assert torch.allclose(ffn_output[0, :2], changed_output[0, :2])
assert all(p.grad is not None for p in ffn.parameters())

修改最后一个 token 后，前两个 token 的输出是否一致： True
up.weight shape= (24, 8) 梯度范数= 0.069103
up.bias shape= (24,) 梯度范数= 0.031633
down.weight shape= (8, 24) 梯度范数= 0.108634
down.bias shape= (8,) 梯度范数= 0.07175


---
## 12. 独立练习（请先停在这里自己实现）

在下面三个代码单元中，只给出函数签名和 TODO。建议用自己的新代码填入，**不要先往下翻答案**。

- **练习 A：统计参数量。** 写 `count_parameters_ex(model)`，返回模型里所有参数元素数量。`XORMLP(16)` 应得到 65。
- **练习 B：三层 MLP。** 写 `ThreeLayerMLPEx`，结构是 `2→8→8→1`，两次 ReLU。输入 `[4,2]` 应输出 `[4,1]`。
- **练习 C：二分类推理。** 写 `predict_binary_ex(model,x)`，使用 `eval()` 和 `no_grad()`，先 sigmoid 再按 0.5 阈值返回 0/1。不要在函数中训练模型。

In [ ]:
# 练习 A：完成后自行测试 count_parameters_ex(XORMLP(16)) == 65

def count_parameters_ex(model):
    # TODO: 遍历 model.parameters()，累加各张量的元素数。
    pass

In [ ]:
# 练习 B：完成后自行测试 ThreeLayerMLPEx()(torch.zeros(4, 2)).shape == (4, 1)

class ThreeLayerMLPEx(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 定义两个隐藏层与输出层。

    def forward(self, x):
        # TODO: 完成前向计算。
        pass

In [ ]:
# 练习 C：完成后自行测试输出是 0/1，且形状为 [N,1]

def predict_binary_ex(model, x):
    # TODO: model.eval()；no_grad；sigmoid；阈值 0.5。
    pass

## 13. 参考答案与自动验收

如果已经完成练习，可以把自己的结果同下面的实现比较。答案用了新名字，**不会覆盖你的练习函数**。

In [19]:
def count_parameters_solution(model):
    return sum(param.numel() for param in model.parameters())


class ThreeLayerMLPSolution(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 8)
        self.layer2 = nn.Linear(8, 8)
        self.layer3 = nn.Linear(8, 1)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        return self.layer3(x)


def predict_binary_solution(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probabilities = torch.sigmoid(logits)
        return (probabilities >= 0.5).to(torch.int64)

print("练习 A 参考结果：", count_parameters_solution(XORMLP(16)))
print("练习 B 参考输出 shape：", tuple(ThreeLayerMLPSolution()(torch.zeros(4, 2)).shape))
print("练习 C 参考预测：", predict_binary_solution(trained_mlp, corners).squeeze(-1).tolist())

练习 A 参考结果： 65
练习 B 参考输出 shape： (4, 1)
练习 C 参考预测： [0, 1, 1, 0]


In [20]:
# 参考答案的基本自动检查：练习参考解不依赖本地脚本文件。
assert count_parameters_solution(XORMLP(16)) == 65
three_layer_model = ThreeLayerMLPSolution()
assert three_layer_model(torch.zeros(4, 2)).shape == (4, 1)
assert count_parameters_solution(three_layer_model) == 105
answers_pred = predict_binary_solution(trained_mlp, corners)
assert answers_pred.shape == (4, 1)
assert answers_pred.squeeze(-1).tolist() == [0, 1, 1, 0]
assert answers_pred.dtype == torch.int64
print("参考答案的固定检查全部通过。")

参考答案的固定检查全部通过。


## 14. 自测问题

请不看答案口头或写下来回答：

1. `nn.Linear(5,12)` 的 `weight` 形状是什么？输入 `[3,7,5]` 时输出形状是什么？
2. 为什么连续叠加两层 `nn.Linear`、中间没有激活函数，不能像带 ReLU 的 MLP 那样解决 XOR？
3. 为什么 `BCEWithLogitsLoss` 不能把先经过 sigmoid 的概率当作 logits 输入？
4. `model.eval()` 与 `torch.no_grad()` 分别改变什么？
5. 如果输入是 `[B,T,D]`，一个 `D→4D→D` 的 FFN 是否改变 token 数量？它会自动混合不同 token 吗？

**参考要点：**

1. `weight=(12,5)`，输出 `[3,7,12]`。
2. 不带非线性的仿射变换可以合并，不能表达 XOR 所需的非线性边界。
3. 该损失函数内部已经完成 sigmoid 相关的稳定计算，输入必须是原始 logits。
4. `eval()` 调整 Dropout、BatchNorm 等层的行为；`no_grad()` 关闭梯度记录。它们不互相替代。
5. 形状仍为 `[B,T,D]`；单独的逐位置 FFN 不会跨位置混合信息。

**下一课方向：** 张量自动求导与模型模块已经会用，下一步可以实现更接近真实语言模型训练任务的 **Embedding、token ID、logits 与交叉熵**，让文字进入神经网络。

---
### 官方资料（按需查阅）

- [PyTorch `nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [PyTorch `nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- [PyTorch `nn.BCEWithLogitsLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [PyTorch 模型保存/加载入门](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html)

本课代码为教学原创示例；**演示用 XOR 数据不等同于语言模型训练**，但梯度、模块和前馈计算是后续课程共用的基础。